# Model Training & Evaluation - Baseline vs Transformer

This notebook trains and evaluates text classification models on the **AG News** dataset:

- **Part A**: TF-IDF + Logistic Regression / Linear SVM baselines
- **Part B**: DistilBERT fine-tuned transformer (conceptual walkthrough)
- **Part C**: Model comparison, inference demo, and error analysis

All models are evaluated on the same test set for fair comparison.

## 1. Imports & Setup

In [ ]:
import sys
import os
import warnings
import time

# Add project root to path for src imports
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import cross_val_score

# Plot styling
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100
warnings.filterwarnings("ignore")

print("Setup complete.")

## 2. Load Data

In [ ]:
from src.data_loader import load_ag_news, create_tfidf_features, LABEL_NAMES

train_df, test_df = load_ag_news()

LABEL_LIST = ["World", "Sports", "Business", "Sci/Tech"]

print(f"Training samples: {len(train_df):,}")
print(f"Test samples:     {len(test_df):,}")
print(f"Labels: {LABEL_NAMES}")

---

# Part A: Baseline Models

## 3. TF-IDF Feature Extraction

In [ ]:
# Create TF-IDF features using the project's data_loader module
tfidf_data = create_tfidf_features(train_df, test_df, max_features=50000, ngram_range=(1, 2))

X_train = tfidf_data["X_train"]
X_test = tfidf_data["X_test"]
y_train = tfidf_data["y_train"]
y_test = tfidf_data["y_test"]
vectorizer = tfidf_data["vectorizer"]

print(f"TF-IDF training matrix: {X_train.shape}")
print(f"TF-IDF test matrix:     {X_test.shape}")
print(f"Features (vocabulary):  {X_train.shape[1]:,}")
print(f"Sparsity: {1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1]):.4%}")

## 4. Logistic Regression

In [ ]:
from src.model import build_baseline_model

# Build Logistic Regression model
lr_model = build_baseline_model("logistic_regression")

# Cross-validation on training set
print("Running 5-fold cross-validation...")
t0 = time.time()
cv_scores = cross_val_score(lr_model, X_train, y_train, cv=5, scoring="accuracy", n_jobs=-1)
cv_time = time.time() - t0

print(f"\nLogistic Regression - Cross-Validation Results:")
print(f"  Mean accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"  Fold scores:   {[f'{s:.4f}' for s in cv_scores]}")
print(f"  CV time:       {cv_time:.1f}s")

In [ ]:
# Train on full training set and evaluate on test set
t0 = time.time()
lr_model.fit(X_train, y_train)
lr_train_time = time.time() - t0

lr_y_pred = lr_model.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_y_pred)
lr_f1 = f1_score(y_test, lr_y_pred, average="macro")

print(f"\nLogistic Regression - Test Results:")
print(f"  Accuracy:   {lr_accuracy:.4f}")
print(f"  F1 (macro): {lr_f1:.4f}")
print(f"  Train time: {lr_train_time:.2f}s")

print(f"\nClassification Report:")
print(classification_report(y_test, lr_y_pred, target_names=LABEL_LIST))

## 5. Linear SVM

In [ ]:
# Build Linear SVM model
svm_model = build_baseline_model("svm")

# Cross-validation
print("Running 5-fold cross-validation...")
t0 = time.time()
svm_cv_scores = cross_val_score(svm_model, X_train, y_train, cv=5, scoring="accuracy", n_jobs=-1)
svm_cv_time = time.time() - t0

print(f"\nLinear SVM - Cross-Validation Results:")
print(f"  Mean accuracy: {svm_cv_scores.mean():.4f} (+/- {svm_cv_scores.std():.4f})")
print(f"  Fold scores:   {[f'{s:.4f}' for s in svm_cv_scores]}")
print(f"  CV time:       {svm_cv_time:.1f}s")

In [ ]:
# Train on full training set and evaluate on test set
t0 = time.time()
svm_model.fit(X_train, y_train)
svm_train_time = time.time() - t0

svm_y_pred = svm_model.predict(X_test)
svm_accuracy = accuracy_score(y_test, svm_y_pred)
svm_f1 = f1_score(y_test, svm_y_pred, average="macro")

print(f"\nLinear SVM - Test Results:")
print(f"  Accuracy:   {svm_accuracy:.4f}")
print(f"  F1 (macro): {svm_f1:.4f}")
print(f"  Train time: {svm_train_time:.2f}s")

print(f"\nClassification Report:")
print(classification_report(y_test, svm_y_pred, target_names=LABEL_LIST))

## 6. Baseline Confusion Matrices

In [ ]:
from src.evaluate import TextClassificationEvaluator

evaluator = TextClassificationEvaluator(label_names=LABEL_LIST)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Logistic Regression confusion matrix
cm_lr = confusion_matrix(y_test, lr_y_pred, normalize="true")
sns.heatmap(
    cm_lr, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=LABEL_LIST, yticklabels=LABEL_LIST, ax=axes[0],
    linewidths=0.5
)
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")
axes[0].set_title(f"Logistic Regression (Acc: {lr_accuracy:.3f})", fontweight="bold")

# SVM confusion matrix
cm_svm = confusion_matrix(y_test, svm_y_pred, normalize="true")
sns.heatmap(
    cm_svm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=LABEL_LIST, yticklabels=LABEL_LIST, ax=axes[1],
    linewidths=0.5
)
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")
axes[1].set_title(f"Linear SVM (Acc: {svm_accuracy:.3f})", fontweight="bold")

plt.tight_layout()
plt.show()

print("Observation: Off-diagonal values show which classes are most confused.")
print("World <-> Business typically shows the highest misclassification rate.")

---

# Part B: DistilBERT Transformer

Fine-tuning DistilBERT requires GPU resources and takes ~15-30 minutes. This section provides a conceptual walkthrough of the pipeline with executable code snippets.

## 7. Tokenization Pipeline

In [ ]:
try:
    from transformers import AutoTokenizer
    HAS_TRANSFORMERS = True
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    print(f"Tokenizer loaded: distilbert-base-uncased")
    print(f"Vocabulary size: {tokenizer.vocab_size:,}")
except ImportError:
    HAS_TRANSFORMERS = False
    print("transformers library not installed. Showing conceptual walkthrough.")

In [ ]:
# Demonstrate the tokenization pipeline that create_tokenized_datasets() uses
sample_text = "Federal Reserve signals potential interest rate cut for next quarter"

if HAS_TRANSFORMERS:
    encoded = tokenizer(
        sample_text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt",
    )

    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    non_pad = [t for t in tokens if t != "[PAD]"]

    print(f"Input text:      {sample_text}")
    print(f"Tokens:          {non_pad}")
    print(f"Number of tokens: {len(non_pad)} (including [CLS] and [SEP])")
    print(f"\nInput IDs shape:      {encoded['input_ids'].shape}")
    print(f"Attention mask shape: {encoded['attention_mask'].shape}")
    print(f"\nThe tokenized dataset is created with:")
    print("  from src.data_loader import create_tokenized_datasets")
    print("  tok_data = create_tokenized_datasets(train_df, test_df)")
    print("  # Returns dict with 'train', 'val', 'test' HF Datasets + 'tokenizer'")
else:
    print("Tokenization pipeline (src.data_loader.create_tokenized_datasets):")
    print("  1. Load AutoTokenizer.from_pretrained('distilbert-base-uncased')")
    print("  2. Split train into train (90%) + validation (10%)")
    print("  3. Tokenize: padding='max_length', truncation=True, max_length=128")
    print("  4. Set format to PyTorch tensors: ['input_ids', 'attention_mask', 'label']")
    print("  5. Returns {'train': Dataset, 'val': Dataset, 'test': Dataset, 'tokenizer': ...}")

## 8. Model Architecture & Parameter Count

In [ ]:
try:
    from src.model import build_transformer_model, get_model_summary

    model = build_transformer_model(
        model_name="distilbert-base-uncased",
        num_labels=4,
        dropout=0.1,
    )

    summary = get_model_summary(model)

    print("DistilBERT for Sequence Classification")
    print("=" * 50)
    print(f"Total parameters:     {summary['total_parameters']:>12,}")
    print(f"Trainable parameters: {summary['trainable_parameters']:>12,}")
    print(f"Frozen parameters:    {summary['frozen_parameters']:>12,}")

    print(f"\nParameter breakdown by layer:")
    for layer, info in summary["layers"].items():
        print(f"  {layer:25s} {info['total']:>12,} params")

    # Clean up to free memory
    del model
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass

except Exception as e:
    print(f"Could not load transformer model ({e}).")
    print("\nDistilBERT Architecture Summary (from README):")
    print("  - 6 transformer layers (vs 12 in BERT-base)")
    print("  - 768 hidden dimensions")
    print("  - 12 attention heads")
    print("  - ~66M total parameters")
    print("  - Classification head: Linear(768, 4)")

## 9. Training Configuration

In [ ]:
from src.utils import DEFAULT_CONFIG

# Display the training configuration used for fine-tuning
training_config = DEFAULT_CONFIG["training"]
model_config = DEFAULT_CONFIG["model"]

print("DistilBERT Fine-Tuning Configuration")
print("=" * 45)
print(f"Model:          {model_config['transformer']}")
print(f"Num labels:     {model_config['num_labels']}")
print(f"Dropout:        {model_config['dropout']}")
print()
print(f"Epochs:         {training_config['epochs']}")
print(f"Batch size:     {training_config['batch_size']}")
print(f"Learning rate:  {training_config['learning_rate']}")
print(f"Weight decay:   {training_config['weight_decay']}")
print(f"Warmup ratio:   {training_config['warmup_ratio']}")
print(f"Seed:           {training_config['seed']}")
print()
print("Training code:")
print("  python src/train.py --model distilbert --epochs 3 --batch_size 32")
print()
print("Or programmatically:")
print("  from src.train import train_transformer")
print("  model, metrics = train_transformer(model, tok_data, num_epochs=3)")

## 10. Expected DistilBERT Results

The following results are from fine-tuning DistilBERT for 3 epochs on the full AG News training set (from the project README).

In [ ]:
# Expected results from README (based on full training)
expected_results = pd.DataFrame({
    "Model": ["TF-IDF + Logistic Regression", "TF-IDF + SVM", "DistilBERT (fine-tuned, 3 epochs)"],
    "Accuracy": [0.912, 0.918, 0.946],
    "F1 (macro)": [0.912, 0.918, 0.946],
    "Inference (ms/sample)": [0.3, 0.5, 8.2],
    "Parameters": ["~400K", "~400K", "66M"],
})

print("Expected Results (from README):")
print()
print(expected_results.to_string(index=False))

print("\n\nKey takeaway: DistilBERT achieves +3.4% accuracy over the best baseline")
print("with only 3 epochs of fine-tuning, at the cost of ~16x slower inference.")

---

# Part C: Model Comparison & Analysis

## 11. Model Comparison

In [ ]:
# Collect actual baseline results alongside expected transformer results
comparison = {
    "TF-IDF + LogReg": {
        "accuracy": lr_accuracy,
        "f1_macro": lr_f1,
        "train_time": lr_train_time,
        "inference_ms": 0.3,   # Approximate per-sample
        "parameters": "~400K",
    },
    "TF-IDF + SVM": {
        "accuracy": svm_accuracy,
        "f1_macro": svm_f1,
        "train_time": svm_train_time,
        "inference_ms": 0.5,
        "parameters": "~400K",
    },
    "DistilBERT": {
        "accuracy": 0.946,     # From README
        "f1_macro": 0.946,
        "train_time": 900.0,   # ~15 min on GPU
        "inference_ms": 8.2,
        "parameters": "66M",
    },
}

comp_df = pd.DataFrame(comparison).T
comp_df.index.name = "Model"
print("Model Comparison Summary:")
print()
print(comp_df.to_string())

In [ ]:
# Bar chart comparison
model_names = list(comparison.keys())
accuracies = [comparison[m]["accuracy"] for m in model_names]
f1_scores = [comparison[m]["f1_macro"] for m in model_names]
colors_bar = ["#4C72B0", "#55A868", "#C44E52"]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Accuracy & F1 comparison
x = np.arange(len(model_names))
width = 0.35
bars1 = axes[0].bar(x - width/2, accuracies, width, label="Accuracy", color="#4C72B0", alpha=0.85)
bars2 = axes[0].bar(x + width/2, f1_scores, width, label="F1 (macro)", color="#C44E52", alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, rotation=10, ha="right")
axes[0].set_ylabel("Score")
axes[0].set_title("Model Performance Comparison", fontweight="bold")
axes[0].legend()
axes[0].set_ylim(0.85, 0.97)
# Add value labels
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f"{bar.get_height():.3f}", ha="center", fontsize=9, fontweight="bold")
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f"{bar.get_height():.3f}", ha="center", fontsize=9, fontweight="bold")

# Inference speed comparison
inf_speeds = [comparison[m]["inference_ms"] for m in model_names]
bars3 = axes[1].bar(model_names, inf_speeds, color=colors_bar, alpha=0.85, edgecolor="white", linewidth=1.2)
axes[1].set_ylabel("Inference Latency (ms/sample)")
axes[1].set_title("Inference Speed Comparison", fontweight="bold")
for bar in bars3:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
                 f"{bar.get_height():.1f} ms", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

print("\nAccuracy vs Speed Tradeoff:")
print(f"  - DistilBERT is {inf_speeds[2]/inf_speeds[0]:.0f}x slower than LogReg at inference")
print(f"  - But achieves +{(accuracies[2] - accuracies[0])*100:.1f} percentage points higher accuracy")
print(f"  - For latency-sensitive applications, TF-IDF baselines remain competitive")
print(f"  - For maximum accuracy, DistilBERT is the clear winner")

## 12. Inference Demo

Show predictions on sample news headlines using the baseline models.

In [ ]:
# Sample news headlines for inference demo
sample_headlines = [
    "Apple unveils new MacBook with M4 chip at developer conference",
    "US women's soccer team wins Olympic gold in thrilling penalty shootout",
    "Federal Reserve raises interest rates by 25 basis points amid inflation",
    "NASA's James Webb telescope discovers high-redshift galaxy candidates",
    "Trade tensions escalate as new tariffs imposed on imported goods",
    "Champions League final draws record 400 million television viewers",
    "Startup raises billion-dollar funding round for AI technology platform",
    "United Nations peacekeepers deployed to conflict region in eastern Africa",
]

# Predict with Logistic Regression
X_headlines = vectorizer.transform(sample_headlines)
lr_preds = lr_model.predict(X_headlines)
lr_proba = lr_model.predict_proba(X_headlines)

print("Inference Demo - Logistic Regression Predictions")
print("=" * 75)
for text, pred, proba in zip(sample_headlines, lr_preds, lr_proba):
    label = LABEL_NAMES[pred]
    conf = proba[pred]
    print(f"\nText: {text}")
    print(f"  Prediction: {label} (confidence: {conf:.3f})")
    print(f"  All scores: ", end="")
    for i, name in LABEL_NAMES.items():
        bar = '|' * int(proba[i] * 30)
        print(f"  {name}: {proba[i]:.3f} {bar}")
        if i < 3:
            print(f"              ", end="")

In [ ]:
# Confidence score visualization for sample predictions
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

colors_conf = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

for idx, (ax, text, proba) in enumerate(zip(axes.flat, sample_headlines, lr_proba)):
    bars = ax.barh(LABEL_LIST, proba, color=colors_conf, alpha=0.8)
    pred_idx = np.argmax(proba)
    bars[pred_idx].set_edgecolor("black")
    bars[pred_idx].set_linewidth(2)
    ax.set_xlim(0, 1)
    ax.set_title(f"{text[:40]}...", fontsize=9, fontweight="bold")
    for i, v in enumerate(proba):
        ax.text(v + 0.01, i, f"{v:.2f}", va="center", fontsize=8)

plt.suptitle("Prediction Confidence Scores (Logistic Regression)", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

## 13. Error Analysis

In [ ]:
# Identify most confused class pairs from the Logistic Regression confusion matrix
cm_raw = confusion_matrix(y_test, lr_y_pred)

# Find off-diagonal entries (misclassifications)
confused_pairs = []
for i in range(4):
    for j in range(4):
        if i != j:
            confused_pairs.append({
                "True Class": LABEL_LIST[i],
                "Predicted As": LABEL_LIST[j],
                "Count": cm_raw[i, j],
                "Error Rate": cm_raw[i, j] / cm_raw[i].sum(),
            })

confused_df = pd.DataFrame(confused_pairs).sort_values("Count", ascending=False)

print("Most Confused Class Pairs (Logistic Regression):")
print()
print(confused_df.head(8).to_string(index=False))

# Highlight the top confusion
top = confused_df.iloc[0]
print(f"\nWorst confusion: {top['True Class']} misclassified as {top['Predicted As']}")
print(f"  {top['Count']} samples ({top['Error Rate']:.1%} of true {top['True Class']} samples)")

In [ ]:
# Show example misclassified texts
test_df_eval = test_df.copy()
test_df_eval["predicted"] = lr_y_pred
test_df_eval["true_name"] = test_df_eval["label"].map(LABEL_NAMES)
test_df_eval["pred_name"] = test_df_eval["predicted"].map(LABEL_NAMES)
test_df_eval["correct"] = test_df_eval["label"] == test_df_eval["predicted"]

misclassified = test_df_eval[~test_df_eval["correct"]]

print(f"Total misclassified: {len(misclassified)} / {len(test_df_eval)} ({len(misclassified)/len(test_df_eval):.1%})")
print(f"\nExample Misclassified Texts:")
print("=" * 80)

# Show 2 examples from each of the top confused pairs
top_pairs = confused_df.head(4)
for _, row in top_pairs.iterrows():
    true_cls = row["True Class"]
    pred_cls = row["Predicted As"]
    examples = misclassified[
        (misclassified["true_name"] == true_cls) &
        (misclassified["pred_name"] == pred_cls)
    ].head(2)

    print(f"\n[True: {true_cls} -> Predicted: {pred_cls}]")
    for _, ex in examples.iterrows():
        print(f"  \"{ex['text'][:120]}...\"")

In [ ]:
# Per-class error rate visualization
per_class_acc = []
for i, name in enumerate(LABEL_LIST):
    mask = y_test == i
    acc = accuracy_score(y_test[mask], lr_y_pred[mask])
    per_class_acc.append({"Class": name, "Accuracy": acc, "Error Rate": 1 - acc})

pca_df = pd.DataFrame(per_class_acc)

fig, ax = plt.subplots(figsize=(10, 5))
colors_err = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
bars = ax.bar(pca_df["Class"], pca_df["Error Rate"], color=colors_err, alpha=0.85, edgecolor="white")
ax.set_ylabel("Error Rate")
ax.set_title("Per-Class Error Rate (Logistic Regression)", fontweight="bold")
for bar, acc_val in zip(bars, pca_df["Accuracy"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"Acc: {acc_val:.3f}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

print("\nSports and Sci/Tech are easiest to classify.")
print("World and Business have higher error rates due to vocabulary overlap.")

---

## 14. Conclusion: When to Use Baseline vs Transformer

### Use TF-IDF + Logistic Regression / SVM when:
- **Low latency** is critical (0.3-0.5 ms/sample vs 8+ ms)
- **Limited compute resources** (no GPU required, trains in seconds)
- **Interpretability** matters (inspect feature weights directly)
- **Quick prototyping** — baselines are fast to iterate on
- Accuracy around **91-92%** is acceptable for the use case

### Use DistilBERT (fine-tuned) when:
- **Maximum accuracy** is the priority (94.6% on AG News)
- You need better handling of **ambiguous/overlapping categories** (World vs Business)
- **Contextual understanding** matters (word meaning depends on surrounding context)
- GPU resources are available for training and inference
- The 3-4% accuracy improvement justifies the computational cost

### Key Findings
- DistilBERT achieves **+3.4%** accuracy over the best baseline with only 3 epochs
- TF-IDF + LR is **~27x faster** at inference and requires no GPU
- **Sports** and **Sci/Tech** are easiest to classify for all models
- **World** vs **Business** confusion is the main challenge — transformers handle this better due to contextual embeddings
- For production deployment, consider a **cascade approach**: use the fast baseline for high-confidence predictions and route uncertain samples to the transformer